# Gobierno · Comentarios, tags y llaves en Gold

## Propósito

Documentar y gobernar las tablas de la capa Gold desde Unity Catalog: cada tabla lleva su comentario con el grano y el owner, su clasificación y sus llaves primarias y foráneas informativas.

## Flujo

1. Aplicar a cada tabla Gold su **COMMENT** y los tags `clasificacion` y `layer`.
2. Declarar la **llave primaria** de cada tabla que la tiene, solo si la llave es única y no tiene nulos.
3. Declarar las **llaves foráneas informativas** entre hechos y dimensiones.

## Destino

Catálogo `gld_dataco`, esquema `supply_chain`. No modifica datos, solo metadatos.

## Reglas

* Las llaves son informativas: Unity Catalog no las exige al escribir, pero documentan el modelo y alimentan el linaje.
* No hay restricciones `CHECK` en las tablas con máscara de columna, porque Databricks no permite combinarlas.
* El notebook es idempotente: si una restricción ya existe, lo informa y continúa.

## Resultado esperado

Una línea por tabla y acción con el estado `OK`, `OMITIDO` o `YA EXISTE/ERROR`.

In [0]:
G = "gld_dataco.supply_chain"

TABLAS = {
  # tabla: (comentario, clasificación, llave primaria)
  "desc_dim_customers":    ("Dimensión de clientes con historial SCD2. Grano: una fila por versión de cliente. Contiene datos personales. Owner: ingeniería de datos.", "confidencial", "cliente_key"),
  "desc_dim_products":     ("Dimensión de productos con historial SCD2. Grano: una fila por versión de producto. Owner: ingeniería de datos.", "interno", "producto_key"),
  "desc_dim_locations":    ("Dimensión de ubicaciones. Grano: una fila por combinación de market, región, país, estado, ciudad y código postal. Owner: ingeniería de datos.", "interno", "location_key"),
  "desc_dim_fecha":        ("Dimensión de fechas. Grano: una fila por fecha. Owner: ingeniería de datos.", "interno", "fecha_key"),
  "desc_dim_shipping_mode":("Dimensión de modos de envío. Grano: una fila por modo de envío. Owner: ingeniería de datos.", "interno", "shipping_mode_key"),
  "desc_ft_orders":        ("Hechos de pedidos con historial SCD2. Grano: una fila por versión de pedido. Owner: ingeniería de datos.", "interno", "orden_key"),
  "desc_ft_order_details": ("Hechos de líneas de pedido. Grano: una fila por línea de pedido. Owner: ingeniería de datos.", "interno", None),
  "desc_ft_order_summary": ("Resumen por pedido, base única de los indicadores. Owner: ingeniería de datos.", "interno", None),
  "desc_ft_kpi_resumen_ejecutivo": ("Indicadores del resumen ejecutivo, con historial por corte. Owner: ingeniería de datos.", "interno", None),
  "desc_ft_kpi_detalle_region":    ("Indicadores por región, con historial por corte. Owner: ingeniería de datos.", "interno", None),
  "desc_ft_kpi_detalle_orden":     ("Indicadores por orden con historial SCD2. Owner: ingeniería de datos.", "interno", "orden_detalle_key"),
  "desc_ft_kpi_alertas":           ("Alertas de indicadores (snapshot por corte). Owner: ingeniería de datos.", "interno", None),
}

FKS = [
  ("desc_ft_orders", "cliente_key", "desc_dim_customers", "cliente_key"),
  ("desc_ft_orders", "location_key", "desc_dim_locations", "location_key"),
  ("desc_ft_orders", "shipping_mode_key", "desc_dim_shipping_mode", "shipping_mode_key"),
  ("desc_ft_orders", "fecha_pedido_key", "desc_dim_fecha", "fecha_key"),
  ("desc_ft_orders", "fecha_envio_key", "desc_dim_fecha", "fecha_key"),
  ("desc_ft_order_details", "orden_key", "desc_ft_orders", "orden_key"),
  ("desc_ft_order_details", "producto_key", "desc_dim_products", "producto_key"),
]

resultados = []
def registrar(t, accion, estado):
    resultados.append((t, accion, estado)); print(f"{estado:<9} {t} · {accion}")

# 1) Comentario y tags
for t, (com, clasif, pk) in TABLAS.items():
    try:
        spark.sql(f"COMMENT ON TABLE {G}.{t} IS '{com}'")
        spark.sql(f"ALTER TABLE {G}.{t} SET TAGS ('clasificacion'='{clasif}', 'layer'='gold')")
        registrar(t, "comentario y tags", "OK")
    except Exception as e:
        registrar(t, "comentario y tags", "ERROR: " + str(e)[:120])

# 2) Llaves primarias (solo si la llave es única y sin nulos)
for t, (com, clasif, pk) in TABLAS.items():
    if not pk: continue
    try:
        if pk not in spark.table(f"{G}.{t}").columns:
            registrar(t, f"PK {pk}", "OMITIDO (columna no existe)"); continue
        n, d, nulos = spark.sql(f"SELECT COUNT(*), COUNT(DISTINCT {pk}), SUM(CASE WHEN {pk} IS NULL THEN 1 ELSE 0 END) FROM {G}.{t}").first()
        if d != n or nulos:
            registrar(t, f"PK {pk}", f"OMITIDO (no única o con nulos: {n} filas, {d} distintas)"); continue
        spark.sql(f"ALTER TABLE {G}.{t} ALTER COLUMN {pk} SET NOT NULL")
        spark.sql(f"ALTER TABLE {G}.{t} ADD CONSTRAINT pk_{t} PRIMARY KEY ({pk})")
        registrar(t, f"PK {pk}", "OK")
    except Exception as e:
        registrar(t, f"PK {pk}", "YA EXISTE/ERROR: " + str(e)[:100])

# 3) Llaves foráneas informativas
for t, col, ref, refcol in FKS:
    try:
        if col not in spark.table(f"{G}.{t}").columns:
            registrar(t, f"FK {col}", "OMITIDO (columna no existe)"); continue
        spark.sql(f"ALTER TABLE {G}.{t} ADD CONSTRAINT fk_{t}_{col} FOREIGN KEY ({col}) REFERENCES {G}.{ref}({refcol})")
        registrar(t, f"FK {col} -> {ref}", "OK")
    except Exception as e:
        registrar(t, f"FK {col} -> {ref}", "YA EXISTE/ERROR: " + str(e)[:100])

# Gobierno · Máscara de datos personales y permisos

## Propósito

Proteger el correo del cliente y dar acceso de lectura al grupo consumidor de Gold.

## Flujo

1. Crear la función `mask_email`: los miembros de `dac_lectores` ven `***@***`; los demás ven el correo real.
2. Marcar la columna `customer_email` con el tag `DAC = YES` y aplicarle la máscara.
3. Dar al grupo `dac_lectores` permiso de lectura (`USE CATALOG`, `USE SCHEMA` y `SELECT`) sobre Gold.

## Destino

`gld_dataco.supply_chain.desc_dim_customers`, columna `customer_email`.

## Control

Después de aplicar la máscara, se ejecuta `gld_dim_customers` para comprobar que el pipeline puede seguir escribiendo en la tabla. Para revertir la máscara: `ALTER TABLE gld_dataco.supply_chain.desc_dim_customers ALTER COLUMN customer_email DROP MASK`.

## Resultado esperado

Los mensajes `OK máscara y tag DAC en customer_email` y `OK permisos para dac_lectores`.

In [0]:
G = "gld_dataco.supply_chain"

# Máscara del correo: los miembros de dac_lectores ven el correo oculto
spark.sql(f"""
CREATE OR REPLACE FUNCTION {G}.mask_email(email STRING)
RETURNS STRING
RETURN CASE WHEN is_account_group_member('dac_lectores') THEN '***@***' ELSE email END
""")

cols = spark.table(f"{G}.desc_dim_customers").columns
assert "customer_email" in cols, "No existe customer_email en desc_dim_customers"

spark.sql(f"ALTER TABLE {G}.desc_dim_customers ALTER COLUMN customer_email SET TAGS ('DAC' = 'YES')")
spark.sql(f"ALTER TABLE {G}.desc_dim_customers ALTER COLUMN customer_email SET MASK {G}.mask_email")
print("OK máscara y tag DAC en customer_email")

# Permisos: el grupo lector consulta Gold
spark.sql("GRANT USE CATALOG ON CATALOG gld_dataco TO `dac_lectores`")
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {G} TO `dac_lectores`")
spark.sql(f"GRANT SELECT ON SCHEMA {G} TO `dac_lectores`")
print("OK permisos para dac_lectores")

In [0]:
# ============================================================
# CELDA 3 · Comentario por columna en Silver y Gold
# Propósito: documentar cada columna con su significado de negocio.
# Cómo funciona: un diccionario nombre_de_columna -> descripción se aplica
#   a todas las tablas donde esa columna exista.
# Resultado: comentarios visibles en Catalog Explorer y en information_schema.columns.
# Idempotente: reaplicar sobreescribe el mismo comentario.
# ============================================================

# DESC: fuente única de verdad de las descripciones.
# Una columna con el mismo nombre en varias tablas recibe el mismo texto.

# Lista de tablas a documentar: 5 de Silver y 12 de Gold, como (esquema, tabla).

# Recorre cada tabla y cada una de sus columnas reales:
#   - si la columna está en DESC: ALTER COLUMN ... COMMENT (se duplican las comillas simples
#     del texto para no romper el SQL).
#   - si no está: se guarda en sin_descripcion para reportarla.
# Un error en una columna se imprime y no detiene el resto.

# Al final imprime cuántos comentarios se aplicaron y la lista de columnas sin descripción,
# para completar DESC y volver a ejecutar.

S = "slv_dataco.supply_chain"
G = "gld_dataco.supply_chain"

DESC = {
 # Clientes
 "customer_id": "Identificador del cliente en el sistema de origen.",
 "customer_fname": "Nombre del cliente, normalizado en mayúsculas.",
 "customer_lname": "Apellido del cliente, normalizado en mayúsculas.",
 "customer_email": "Correo del cliente. Dato personal, enmascarado para dac_lectores. En el dataset público viene anonimizado.",
 "customer_segment": "Segmento del cliente según el origen.",
 "customer_city": "Ciudad del cliente.", "customer_state": "Estado del cliente.",
 "customer_country": "País del cliente.", "customer_street": "Calle del cliente.",
 "customer_zipcode": "Código postal del cliente.",
 # Productos
 "product_card_id": "Identificador del producto.", "product_category_id": "Identificador de la categoría del producto.",
 "category_name": "Nombre de la categoría.", "department_id": "Identificador del departamento.",
 "department_name": "Nombre del departamento.", "product_name": "Nombre del producto.",
 "product_description": "Descripción del producto.", "product_price": "Precio de lista del producto.",
 "product_status": "Estado del producto según el origen.",
 # Ubicación y pedido
 "market": "Mercado del pedido.", "order_region": "Región de destino del pedido.",
 "order_country": "País de destino del pedido.", "order_state": "Estado de destino del pedido.",
 "order_city": "Ciudad de destino del pedido.", "order_zipcode": "Código postal de destino del pedido.",
 "latitude": "Latitud de la ubicación.", "longitude": "Longitud de la ubicación.",
 "grupo_mercado": "Agrupación de mercado usada en los indicadores (AMERICAS agrupa LATAM y USCA).",
 "order_id": "Identificador del pedido.", "order_customer_id": "Identificador del cliente que hizo el pedido.",
 "order_date": "Fecha y hora del pedido.", "shipping_date": "Fecha y hora de envío.",
 "shipping_mode": "Modo de envío.", "delivery_status": "Estado de la entrega según el origen.",
 "order_status": "Estado del pedido según el origen.",
 "days_for_shipping_real": "Días reales de envío.", "days_for_shipment_scheduled": "Días de envío programados.",
 "late_delivery_risk": "Riesgo de retraso según el origen: 1 con riesgo, 0 sin riesgo.",
 # Líneas de pedido
 "order_item_id": "Identificador de la línea de pedido.", "order_item_cardprod_id": "Identificador del producto de la línea.",
 "order_item_quantity": "Cantidad de unidades de la línea.", "order_item_product_price": "Precio unitario del producto en la línea.",
 "order_item_discount": "Monto de descuento de la línea.", "order_item_discount_rate": "Tasa de descuento de la línea, entre 0 y 1.",
 "order_item_profit_ratio": "Razón de beneficio de la línea.", "order_item_total": "Total de la línea después del descuento.",
 "sales": "Ventas de la línea.", "sales_per_customer": "Ventas por cliente, tal como vienen del origen.",
 "order_profit_per_order": "Beneficio del pedido, tal como viene del origen.", "benefit_per_order": "Beneficio por pedido, tal como viene del origen.",
 # Calidad y trazabilidad
 "estado_calidad": "VALIDO o CUARENTENA. Gold solo lee las filas VALIDO.",
 "motivo_cuarentena": "Motivo por el que la fila está en cuarentena; NULL si es válida.",
 "fecha_procesamiento": "Momento en que Silver procesó la fila.",
 "fecha_inicio_corte": "Inicio de la ventana de fechas cargada.", "fecha_fin_corte": "Fin de la ventana de fechas cargada.",
 "fecha_carga": "Momento de carga a Bronze.", "id_ejecucion": "Identificador de la ejecución del pipeline que cargó la fila.",
 # SCD2 y llaves
 "es_vigente": "Verdadero en la versión actual de la fila.",
 "fecha_inicio_vigencia": "Fecha desde la que la versión es vigente.", "fecha_fin_vigencia": "Fecha en que dejó de ser vigente; NULL si es la actual.",
 "cliente_key": "Llave sustituta del cliente (una por versión).", "producto_key": "Llave sustituta del producto (una por versión).",
 "location_key": "Llave sustituta de la ubicación (hash estable).", "shipping_mode_key": "Llave sustituta del modo de envío (hash estable).",
 "fecha_key": "Llave de la fecha.", "orden_key": "Llave sustituta del pedido (una por versión).",
 "fecha_pedido_key": "Llave de la fecha del pedido.", "fecha_envio_key": "Llave de la fecha de envío.",
 "orden_detalle_key": "Llave sustituta del detalle de orden (una por versión).",
 # Calendario
 "fecha": "Fecha del calendario.", "anio": "Año.", "trimestre": "Trimestre.", "mes": "Número de mes.",
 "nombre_mes": "Nombre del mes.", "dia": "Día del mes.", "dia_semana_num": "Número del día de la semana.",
 "nombre_dia": "Nombre del día de la semana.", "semana_anio": "Semana del año.",
 "es_fin_de_semana": "Verdadero si es sábado o domingo.",
 # Resumen por pedido
 "total_sales": "Suma de ventas del pedido.", "total_order_item": "Suma de los totales de línea del pedido.",
 "total_profit": "Suma del beneficio del pedido.", "avg_benefit_per_order": "Promedio del beneficio por orden.",
 "total_quantity": "Suma de unidades del pedido.", "total_lineas": "Cantidad de líneas distintas del pedido.",
 "es_entrega_a_tiempo": "Verdadero si el pedido se entregó a tiempo.",
 "es_excluido_otd_leadtime": "Verdadero si el pedido se excluye del cálculo de On-Time Delivery y Lead Time.",
 "late_delivery_risk_calculado": "Riesgo de retraso validado en Gold.",
  "late_delivery_risk_original": "Riesgo de retraso tal como viene del origen: 1 con riesgo, 0 sin riesgo.",
 "riesgo_validado": "Verdadero si el riesgo del origen coincide con el riesgo calculado en Gold; falso indica una discrepancia.",
 "motivo_alerta": "Motivo por el que el pedido está en alertas: RIESGO_NO_VALIDADO, FRAUDE_SOSPECHOSO o RIESGO_NO_VALIDADO_Y_FRAUDE_SOSPECHOSO.",
 "shipping_variance": "Variación entre los días reales y los programados de envío.",
 "es_sospechoso_fraude": "Verdadero si el pedido está marcado como sospechoso de fraude.",
 # Indicadores
 "grupo": "Mercado, AMERICAS o GLOBAL al que corresponde el indicador.",
 "fecha_snapshot": "Fecha del corte del indicador.", "es_ultimo_snapshot": "Verdadero en el snapshot más reciente.",
 "on_time_delivery_pct": "Proporción de pedidos entregados a tiempo, sin los excluidos.",
 "late_delivery_risk_pct": "Proporción de pedidos con riesgo de retraso.",
 "shipping_variance_avg": "Promedio de la variación de envío, sin los excluidos.",
 "profit_margin_pct": "Beneficio total entre ventas totales, por 100.",
 "revenue_por_cliente": "Ventas totales entre clientes distintos.",
 "beneficio_por_orden": "Promedio del beneficio por orden.",
 "lead_time_promedio": "Promedio de días reales de envío, sin los excluidos.",
 "total_sospechosos_fraude": "Cantidad de pedidos marcados como sospechosos de fraude.",
 "total_pedidos": "Cantidad de pedidos del grupo.",
}

tablas = [(S, t) for t in ["md_clientes","md_productos","md_ubicaciones","hd_pedidos","hd_pedido_detalle"]] + \
         [(G, t) for t in ["desc_dim_customers","desc_dim_products","desc_dim_locations","desc_dim_fecha","desc_dim_shipping_mode",
                           "desc_ft_orders","desc_ft_order_details","desc_ft_order_summary",
                           "desc_ft_kpi_resumen_ejecutivo","desc_ft_kpi_detalle_region","desc_ft_kpi_detalle_orden","desc_ft_kpi_alertas"]]

sin_descripcion, aplicadas = [], 0
for esq, t in tablas:
    for c in spark.table(f"{esq}.{t}").columns:
        if c in DESC:
            txt = DESC[c].replace("'", "''")
            try:
                spark.sql(f"ALTER TABLE {esq}.{t} ALTER COLUMN `{c}` COMMENT '{txt}'"); aplicadas += 1
            except Exception as e:
                print("ERROR", t, c, str(e)[:100])
        else:
            sin_descripcion.append((t, c))
print(f"✓ Comentarios aplicados: {aplicadas}")
print(f"Columnas sin descripción: {len(sin_descripcion)}")
for t, c in sin_descripcion: print("  ", t, c)

In [0]:
# ============================================================
# CELDA 4 · Diccionario de datos consultable
# Propósito: tener una tabla con una fila por columna de cada tabla de las tres capas.
# Fuente: Unity Catalog (information_schema), no un listado escrito a mano;
#   por eso nunca se desactualiza: se regenera cada vez.
# Destino: metadata_dataco.supply_chain.md_diccionario_datos
# ============================================================

# Para cada catálogo (Bronze, Silver, Gold) consulta:
#   - columns: nombre, posición, tipo y comentario de la columna.
#   - tables: comentario de la tabla (se excluyen vistas y el esquema information_schema).
#   - column_tags: el tag DAC; es_dato_personal = true si DAC = 'YES'.

# Une los tres resultados con unionByName y agrega fecha_generacion.

# Escribe con overwrite + overwriteSchema: la tabla se reconstruye completa
#   en cada ejecución, sin duplicados.

# Le pone comentario y tags (layer, domain, owner) a la propia tabla del diccionario.

# Imprime el total de columnas documentadas y, por tabla, cuántas siguen sin descripción:
#   es el indicador para saber qué falta documentar.
from pyspark.sql import functions as F

partes = []
for cat in ["brz_dataco", "slv_dataco", "gld_dataco"]:
    partes.append(spark.sql(f"""
      SELECT c.table_catalog AS catalogo, c.table_schema AS esquema, c.table_name AS tabla,
             c.column_name AS columna, c.ordinal_position AS posicion, c.data_type AS tipo,
             c.comment AS descripcion, t.comment AS descripcion_tabla,
             CASE WHEN d.tag_value = 'YES' THEN true ELSE false END AS es_dato_personal
      FROM {cat}.information_schema.columns c
      JOIN {cat}.information_schema.tables t
        ON c.table_schema = t.table_schema AND c.table_name = t.table_name
      LEFT JOIN {cat}.information_schema.column_tags d
        ON c.table_schema = d.schema_name AND c.table_name = d.table_name
       AND c.column_name = d.column_name AND d.tag_name = 'DAC'
      WHERE c.table_schema <> 'information_schema' AND t.table_type <> 'VIEW'
    """))

df = partes[0]
for p in partes[1:]: df = df.unionByName(p)
df = df.withColumn("fecha_generacion", F.current_timestamp())

DICC = "metadata_dataco.supply_chain.md_diccionario_datos"
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(DICC)
spark.sql(f"COMMENT ON TABLE {DICC} IS 'Diccionario de datos de las tres capas, generado desde Unity Catalog. Grano: una fila por columna de cada tabla. Se regenera al ejecutar aplicar_gobierno. Owner: ingeniería de datos.'")
spark.sql(f"ALTER TABLE {DICC} SET TAGS ('layer'='metadata','domain'='supply_chain','owner'='ingenieria_de_datos')")
print("✓ Diccionario:", spark.table(DICC).count(), "columnas documentadas")
display(spark.sql(f"SELECT tabla, COUNT(*) columnas, SUM(CASE WHEN descripcion IS NULL THEN 1 ELSE 0 END) sin_descripcion FROM {DICC} GROUP BY tabla ORDER BY tabla"))

In [0]:
# ============================================================
# CELDA 5 · Gobierno de Silver y Bronze
# Prerrequisito: grupo dac_analistas_avanzados creado en Settings.
# Idempotente: se puede ejecutar varias veces sin efectos distintos.
# ============================================================
S = "slv_dataco.supply_chain"
SILVER = {
 "md_clientes": ("Maestro de clientes. Grano: una fila por cliente. Contiene datos personales. Las filas que incumplen una regla quedan en cuarentena. Owner: ingeniería de datos.", "confidencial"),
 "md_productos": ("Maestro de productos. Grano: una fila por producto. Owner: ingeniería de datos.", "interno"),
 "md_ubicaciones": ("Maestro de ubicaciones. Grano: una fila por combinación geográfica. Owner: ingeniería de datos.", "interno"),
 "hd_pedidos": ("Pedidos. Grano: una fila por pedido. Las filas que incumplen una regla quedan en cuarentena. Owner: ingeniería de datos.", "interno"),
 "hd_pedido_detalle": ("Líneas de pedido. Grano: una fila por línea de pedido. Las filas que incumplen una regla quedan en cuarentena. Owner: ingeniería de datos.", "interno"),
}
for t, (com, clasif) in SILVER.items():
    spark.sql(f"COMMENT ON TABLE {S}.{t} IS '{com}'")
    spark.sql(f"ALTER TABLE {S}.{t} SET TAGS ('clasificacion'='{clasif}','layer'='silver')")
print("✓ Comentarios y tags de Silver")

spark.sql("ALTER TABLE brz_dataco.sharepoint.raw_supply_chain SET TAGS ('clasificacion'='confidencial','layer'='bronze')")
print("✓ Tags de Bronze")

# Misma máscara de correo en Silver
spark.sql(f"""CREATE OR REPLACE FUNCTION {S}.mask_email(email STRING) RETURNS STRING
RETURN CASE WHEN is_account_group_member('dac_lectores') OR is_account_group_member('dac_analistas_avanzados') THEN '***@***' ELSE email END""")
spark.sql(f"ALTER TABLE {S}.md_clientes ALTER COLUMN customer_email SET TAGS ('DAC'='YES')")
spark.sql(f"ALTER TABLE {S}.md_clientes ALTER COLUMN customer_email SET MASK {S}.mask_email")
print("✓ Máscara en Silver")

# Permisos de Silver para analistas avanzados (Bronze: sin permisos a grupos, solo ingeniería)
spark.sql("GRANT USE CATALOG ON CATALOG slv_dataco TO `dac_analistas_avanzados`")
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {S} TO `dac_analistas_avanzados`")
spark.sql(f"GRANT SELECT ON SCHEMA {S} TO `dac_analistas_avanzados`")
print("✓ Permisos de Silver")